# ShopBot — QLoRA Qwen3-4B-Instruct-2507 (router + chọn bằng chứng)

1. **Settings → Accelerator: GPU T4 x2** (dùng 1 GPU), **Internet: On**.
2. **Add Input** → dataset `shopbot-qwen-sft` (thư mục `data/finetune_qwen_20261026`: `train.jsonl`, `dev.jsonl`, `DATA_CARD.json`).
3. **Run All** (hoặc Save Version → Save & Run All để chạy khi tắt trình duyệt).

Kết quả trong `/kaggle/working`: `metrics.json`, `lora_adapter/`, `gguf/*.gguf` (q4_k_m). Tải file `.gguf` về máy rồi chạy `scripts/import_qwen_sft_ollama.ps1`.

Nhãn được dựng từ database/tài liệu shop, không lấy câu trả lời của bot; dev tách theo sản phẩm/tài liệu. Điểm ở đây chỉ là khớp JSON trên dev; đánh giá thật là bộ 479/121 chạy lại trên máy.

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import glob, json, os, random, time, torch
SEED=20261026; random.seed(SEED)
MAX_LEN=4096
root=os.path.dirname(sorted(glob.glob('/kaggle/input/**/train.jsonl',recursive=True))[0])
train=[json.loads(l) for l in open(f'{root}/train.jsonl',encoding='utf-8')]
dev=[json.loads(l) for l in open(f'{root}/dev.jsonl',encoding='utf-8')]
card=json.load(open(f'{root}/DATA_CARD.json',encoding='utf-8'))
print(root, len(train), len(dev), card['sha256'])
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GB')

In [ ]:
from unsloth import FastLanguageModel
model,tokenizer=FastLanguageModel.from_pretrained('unsloth/Qwen3-4B-Instruct-2507',max_seq_length=MAX_LEN,load_in_4bit=True,dtype=None)
# LoRA B starts at zero, so the 'base' evaluation below is the untouched model.
model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',random_state=SEED)

def prompt(messages):
    # Same text Ollama renders for qwen3:4b-instruct-2507 (no <think> block).
    system,user=messages[0]['content'],messages[1]['content']
    return f'<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n'

def encode(example):
    head=tokenizer(prompt(example['messages']),add_special_tokens=False)['input_ids']
    tail=tokenizer(example['messages'][2]['content']+'<|im_end|>\n',add_special_tokens=False)['input_ids']
    ids=head+tail
    # Loss on the assistant JSON only.
    return dict(input_ids=ids,attention_mask=[1]*len(ids),labels=[-100]*len(head)+tail)

enc_train=[encode(e) for e in train]; enc_dev=[encode(e) for e in dev]
too_long=sum(len(e['input_ids'])>MAX_LEN for e in enc_train)
enc_train=[e for e in enc_train if len(e['input_ids'])<=MAX_LEN]
print('train',len(enc_train),'dropped >MAX_LEN',too_long,'max tokens',max(len(e['input_ids']) for e in enc_train))

In [ ]:
def decision(d):
    # Unconstrained generation (no Ollama schema here) can return any JSON
    # shape; anything not matching the schema simply counts as wrong.
    if not isinstance(d,dict) or not isinstance(d.get('answers'),dict) or not isinstance(d.get('missing'),list):return None
    if not all(isinstance(v,list) for v in d['answers'].values()):return None
    try:return ({str(k):sorted(map(str,v)) for k,v in d['answers'].items()},sorted(map(str,d['missing'])))
    except TypeError:return None

def evaluate(rows,limit=None,tag=''):
    """Greedy JSON generation; exact match of the decision fields."""
    FastLanguageModel.for_inference(model)
    rows=rows[:limit] if limit else rows
    stats=dict(n=0,parsed=0,router_action=0,router_n=0,selector_exact=0,selector_missing=0,selector_n=0)
    start=time.time()
    for e in rows:
        ids=tokenizer(prompt(e['messages']),return_tensors='pt',add_special_tokens=False).to('cuda')
        with torch.no_grad():
            out=model.generate(**ids,max_new_tokens=220,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        text=tokenizer.decode(out[0][ids['input_ids'].shape[1]:],skip_special_tokens=True).strip()
        gold=json.loads(e['messages'][2]['content']); stats['n']+=1
        try:pred=json.loads(text); stats['parsed']+=1
        except Exception:pred={}
        if not isinstance(pred,dict):pred={}
        if e['task']=='router':
            stats['router_n']+=1
            stats['router_action']+=pred.get('action')==gold['action'] and bool(pred.get('private_request'))==gold['private_request']
        else:
            stats['selector_n']+=1
            got=decision(pred)
            stats['selector_exact']+=got==decision(gold)
            stats['selector_missing']+=got is not None and got[1]==sorted(gold['missing'])
    stats['seconds']=round(time.time()-start)
    print(tag,stats); return stats

random.Random(SEED).shuffle(dev)
EVAL_N=120
metrics=dict(data=card['sha256'],base=evaluate(dev,EVAL_N,'base'))

In [ ]:
FastLanguageModel.for_training(model)
from datasets import Dataset
from transformers import Trainer,TrainingArguments,DataCollatorForSeq2Seq
args=TrainingArguments(output_dir='/kaggle/working/run',per_device_train_batch_size=1,gradient_accumulation_steps=8,
    num_train_epochs=2,learning_rate=1e-4,lr_scheduler_type='cosine',warmup_steps=20,weight_decay=0.0,
    fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),logging_steps=10,
    eval_strategy='steps',eval_steps=100,per_device_eval_batch_size=1,save_strategy='no',report_to='none',seed=SEED,optim='adamw_8bit')
trainer=Trainer(model=model,args=args,train_dataset=Dataset.from_list(enc_train),eval_dataset=Dataset.from_list(enc_dev[:80]),
    data_collator=DataCollatorForSeq2Seq(tokenizer,padding=True,label_pad_token_id=-100))
result=trainer.train(); print(result)
metrics['train']=dict(loss=result.training_loss,seconds=round(result.metrics['train_runtime']),log=trainer.state.log_history)

In [ ]:
metrics['tuned']=evaluate(dev,EVAL_N,'tuned')
metrics['tuned_full_dev']=evaluate(dev,None,'tuned full dev')
json.dump(metrics,open('/kaggle/working/metrics.json','w'),indent=1)
model.save_pretrained('/kaggle/working/lora_adapter'); tokenizer.save_pretrained('/kaggle/working/lora_adapter')

In [ ]:
# GGUF q4_k_m for Ollama (same quantization as the current local model).
model.save_pretrained_gguf('/kaggle/working/gguf',tokenizer,quantization_method='q4_k_m')
for path in glob.glob('/kaggle/working/gguf*/**/*.gguf',recursive=True)+glob.glob('/kaggle/working/*.gguf'):
    print(path, round(os.path.getsize(path)/2**30,2),'GB')